# Notebook 4: Stress-Induced Anisotropy in dv/v## Anisotropic stress effects and crack-closure models### Key References- Tromp, J. & Trampert, J. (2018). Effects of induced stress on seismic forward modelling and inversion. *GJI*, 213, 851–867.- Fokker, E. et al. (2021). *Remote Sensing*, 13, 2684.- Okubo, K. et al. (2024). *JGR Solid Earth*, 129, e2023JB028084.- Sayers, C. M. & Kachanov, M. (1995). *JGR*, 100, 4149–4156.### Physical MotivationAt Parkfield, the observed long-term increase in dv/v correlates with the **contractional axial strain** (not dilatational strain), suggesting **anisotropic crack closure**. Cracks aligned perpendicular to the maximum compressive stress close preferentially, increasing rigidity and velocity in that direction (Okubo et al., 2024).

In [ ]:
import numpy as npimport matplotlib.pyplot as pltplt.rcParams.update({    'font.size': 12, 'axes.labelsize': 14, 'figure.dpi': 150,    'font.family': 'serif', 'mathtext.fontset': 'cm',    'axes.grid': True, 'grid.alpha': 0.3})print("Environment ready.")

## 1. Tromp & Trampert (2018) FrameworkWithout higher-order elasticity, induced stress modifies wave speeds through the equation of motion. For S-waves under induced deviatoric stress $\tau^0$:$$\frac{d\beta}{\beta} = \frac{\mu' p^0}{2\mu} + \frac{1-\mu'}{4\mu} \hat{k}\cdot\tau^0\cdot\hat{k} - \frac{1+\mu'}{4\mu} \hat{a}\cdot\tau^0\cdot\hat{a}$$where:- $\hat{k}$ = propagation direction- $\hat{a}$ = polarization direction  - $p^0$ = induced pressure- $\tau^0$ = induced deviatoric stress- $\mu' = d\mu/dP$ = pressure derivative of shear modulusThis shows that **deviatoric stress creates anisotropy** in wave speeds without requiring third-order elasticity — the anisotropy arises from the stress-dependent constitutive relation (Fokker et al., 2021, Eq. 4).

In [ ]:
# === Anisotropic dv/v from Tromp & Trampert (2018) ===def dvv_anisotropic(theta, phi, T33, mu_prime, mu, wave='SV'):    """    Compute dv/v as a function of propagation direction for uniaxial vertical stress.        theta : polar angle from vertical (0 = down) [radians]    phi : azimuthal angle [radians]      T33 : vertical stress [Pa]    mu_prime : shear modulus pressure derivative    mu : shear modulus [Pa]        Returns dv/v for SV and SH waves.    """    # For uniaxial vertical stress T33:    # Deviatoric stress tau_0 = -T33/3 * diag(1, 1, -2)    # Pressure p0 = -T33/3        p0 = -T33 / 3        # Propagation direction    kx = np.sin(theta) * np.cos(phi)    ky = np.sin(theta) * np.sin(phi)    kz = np.cos(theta)        # k . tau . k for the deviatoric part    # tau = -T33/3 * diag(1,1,-2)    k_tau_k = -T33/3 * (kx**2 + ky**2 - 2*kz**2)        # Isotropic pressure term    dvv_iso = mu_prime * p0 / (2 * mu)        # Deviatoric terms depend on polarization    if wave == 'SV':        # SV polarization in the sagittal plane        # a = (-cos(theta)*cos(phi), -cos(theta)*sin(phi), sin(theta))        ax = -np.cos(theta) * np.cos(phi)        ay = -np.cos(theta) * np.sin(phi)        az = np.sin(theta)        a_tau_a = -T33/3 * (ax**2 + ay**2 - 2*az**2)    else:  # SH        # SH perpendicular to sagittal plane        ax = -np.sin(phi)        ay = np.cos(phi)        az = 0.0        a_tau_a = -T33/3 * (ax**2 + ay**2)        dvv = dvv_iso + (1-mu_prime)/(4*mu) * k_tau_k - (1+mu_prime)/(4*mu) * a_tau_a    return dvv# Polar plot of dv/v vs propagation angletheta = np.linspace(0, np.pi, 180)phi = 0  # azimuth fixedT33_values = [-1e6, -5e6, -10e6]  # vertical compression [Pa]mu = 30e9mu_prime = 100fig, axes = plt.subplots(1, 3, figsize=(16, 5))for i, T33 in enumerate(T33_values):    ax = axes[i]    dvv_sv = np.array([dvv_anisotropic(th, phi, T33, mu_prime, mu, 'SV') for th in theta])    dvv_sh = np.array([dvv_anisotropic(th, phi, T33, mu_prime, mu, 'SH') for th in theta])        ax.plot(np.degrees(theta), dvv_sv*1e4, 'b-', lw=2, label='SV')    ax.plot(np.degrees(theta), dvv_sh*1e4, 'r-', lw=2, label='SH')    ax.plot(np.degrees(theta), (dvv_sv-dvv_sh)*1e4, 'k--', lw=1.5, label='SV-SH (splitting)')    ax.set_xlabel('Propagation angle from vertical [°]')    ax.set_ylabel('dv/v [×10⁻⁴]')    ax.set_title(f'$T_{{33}}$ = {T33/1e6:.0f} MPa')    ax.legend(fontsize=9)    ax.axhline(0, color='gray', ls='-', alpha=0.5)fig.suptitle('Directional Dependence of dv/v Under Uniaxial Stress\n'             '(Tromp & Trampert, 2018; Fokker et al., 2021)', fontsize=14, y=1.02)plt.tight_layout()plt.savefig('fig11_anisotropic_dvv.png', bbox_inches='tight')plt.show()

## 2. Microcrack-Induced AnisotropyFollowing Okubo et al. (2024) and Sayers & Kachanov (1995), oriented microcracks create elastic anisotropy. Under tectonic compression along azimuth $\psi$:- Cracks perpendicular to $\psi$ close → velocity increases along $\psi$- Cracks parallel to $\psi$ remain open → velocity unchanged perpendicular to $\psi$This produces **VTI-like anisotropy** with the fast axis aligned with the compression.At Parkfield, the maximum contractional strain is oriented N35°W to N45°E,consistent with SAF right-lateral loading. The rotated axial strain shows compressionin this range, explaining the long-term dv/v increase.

In [ ]:
# === Crack-Induced Anisotropy Model ===def crack_velocity(theta_prop, theta_stress, crack_density, V0, aspect_ratio=0.01):    """    Simple model of velocity vs propagation direction for stress-aligned cracks.        Cracks perpendicular to stress direction close proportionally to stress.        theta_prop : propagation azimuth [degrees]    theta_stress : maximum compressive stress azimuth [degrees]    crack_density : initial crack density parameter    V0 : uncracked velocity [m/s]    """    # Angle between propagation and stress    angle_diff = np.radians(theta_prop - theta_stress)        # Cracks perpendicular to stress close → reduce crack density for that direction    # Effective crack density seen by wave propagating at angle theta    # Cracks with normals along stress direction close → waves along stress are faster    effective_crack = crack_density * np.sin(angle_diff)**2        # Hudson (1981) first-order correction    dV_V = -effective_crack * (8.0/3.0) * (1 + aspect_ratio)        return V0 * (1 + dV_V)# Azimuthal velocity variationazimuths = np.linspace(0, 360, 361)stress_azimuth = -40  # N40°W, roughly Parkfield max compressionfig, axes = plt.subplots(1, 2, figsize=(14, 6))# Panel (a): Velocity rose diagramax = axes[0]crack_densities = [0.0, 0.02, 0.05, 0.10, 0.15]for cd in crack_densities:    V = crack_velocity(azimuths, stress_azimuth, cd, 3000)    ax.plot(azimuths, V, lw=2, label=f'$\\epsilon_c$ = {cd}')ax.axvline(stress_azimuth % 360, color='red', ls='--', alpha=0.7, label='Max compression')ax.set_xlabel('Propagation azimuth [°]')ax.set_ylabel('Velocity [m/s]')ax.set_title('(a) Azimuthal velocity variation from crack closure')ax.legend(fontsize=9)# Panel (b): dv/v as function of tectonic strain at different azimuthsax = axes[1]strain_range = np.linspace(0, 2e-6, 100)  # contractional strainmeasure_azimuths = [0, -40, -90, 45]  # different measurement azimuthsfor az in measure_azimuths:    dvv = []    for strain in strain_range:        cd_eff = 0.1 * (1 - strain / 2e-6)  # cracks close with strain        V = crack_velocity(az, stress_azimuth, cd_eff, 3000)        V0 = crack_velocity(az, stress_azimuth, 0.1, 3000)        dvv.append((V - V0) / V0 * 100)    ax.plot(strain_range*1e6, dvv, lw=2, label=f'Measurement az = {az}°')ax.set_xlabel('Contractional tectonic strain [microstrain]')ax.set_ylabel('dv/v [%]')ax.set_title('(b) dv/v from crack closure vs tectonic strain\n(cf. Okubo et al. 2024, Fig. 14)')ax.legend()fig.suptitle('Microcrack-Induced Anisotropy & Tectonic dv/v\n'             '(Sayers & Kachanov, 1995; Okubo et al., 2024)', fontsize=14, y=1.02)plt.tight_layout()plt.savefig('fig12_crack_anisotropy.png', bbox_inches='tight')plt.show()print("Key finding (Okubo et al., 2024): The long-term dv/v increase at Parkfield")print("correlates with contractional axial strain (N35°W to N45°E) but NOT with dilatation,")print("suggesting anisotropic crack closure as the mechanism.")